# Classification multilabel des PFAS — Protocole Dong et al. (2024)

**Référence :** Dong, J. et al. (2024). *Prediction of 35 Target Per- and Polyfluoroalkyl
Substances (PFASs) in California Groundwater Using Multilabel Semisupervised Machine Learning.*

**Objectif :** Prédire simultanément, pour chaque puits californien, **quels PFAS individuels**
dépassent leur seuil réglementaire — à partir des seules variables environnementales
(hydrogéologie, sol, climat, proximité aux sources), **sans aucune mesure PFAS** en entrée.

## Le défi : un problème *multilabel* et *semi-supervisé*

| Particularité | Conséquence méthodologique |
|---|---|
| **27 PFAS cibles** mesurés simultanément | Problème **multilabel** (≠ 27 problèmes binaires indépendants) |
| Les PFAS **co-occurrent** (mêmes sources) | → **Chaîne de classifieurs** : chaque modèle exploite les prédictions des précédents |
| Tous les programmes ne mesurent **pas tous** les PFAS (NaN) | → **Semi-supervision** : pseudo-étiquetage des échantillons non mesurés |
| Forte **rareté** de certains analytes | → **SMOTE** par label |

## Seuils de détection — réglementation EPA 2024 NPDWR

Chaque PFAS utilise son **MCL individuel EPA 2024** quand il existe, sinon **2 ng/L**
(limite analytique ≈ MDL). Les non-détects sont stockés à MDL/2 ≈ 1 ng/L → label 0.

| PFAS | Seuil | Source |
|---|---|---|
| PFOA, PFOS | **4 ng/L** | MCL EPA 2024 NPDWR |
| PFHxS, PFNA | **10 ng/L** | MCL EPA 2024 NPDWR |
| *(HFPO-DA : MCL 10 ng/L mais max dataset = 4.5 ng/L → exclu)* | — | — |
| 23 autres | 2 ng/L | MDL analytique |

## Retrait de la localisation (protocole Dong)

Comme dans le papier, les **identifiants géographiques purs** (lat/lon, county, bassins)
sont retirés pour forcer l'apprentissage *environnemental* et éviter la mémorisation
spatiale. Les **features de proximité aux sources** (distance/densité des sites PFAS)
sont **conservées** : elles sont mécanistiques.

## 0. Imports et configuration

In [24]:
import sys, json, pickle, warnings, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Accès au package src/
sys.path.insert(0, "..")
# Rechargement forcé : prend en compte les modifs de src/ sans restart kernel
import src.ml_multilabel as _mlm
importlib.reload(_mlm)
from src.ml_multilabel import (
    # configuration
    DETECTION_THRESHOLDS, DEFAULT_THRESHOLD, PFAS_TARGET_COLS,
    LOCATION_FEATURES, CATEGORICAL_FEATURES,
    XGB_PARAMS, XGB_PARAMS_FAST, PSEUDO_HIGH, PSEUDO_LOW,
    RANDOM_STATE, CV_FOLDS, TEST_SIZE,
    # pipeline
    build_label_matrix, build_feature_matrix, build_preprocessor,
    fit_chain, predict_chain, evaluate_multilabel, cross_validate_chain,
    # comparaison des approches de division en classes
    COVERAGE_TIERS, TIER_PSEUDO_POLICY,
    fit_nested_chain, fit_class_chains, predict_class_chains, evaluate_by_tier,
    _label_name, _tier_of,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ── Chemins ──────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"
FIGURES_DIR   = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# ── Paramètres d'exécution (modifiables) ─────────────────────────────────────
#  Pour les RÉSULTATS FINAUX du mémoire : SAMPLE_SIZE = None  et  FAST = False
#  Pour l'exploration interactive rapide : garder les valeurs ci-dessous
SAMPLE_SIZE  = 15_000   # None → dataset complet (46 338) ; sous-échantillon sinon
FAST         = True     # True → XGBoost réduit (100 arbres) pour l'interactivité
DROP_LOCATION = True    # protocole Dong et al. — retrait des identifiants géo
USE_CHAIN    = True     # chaîne de classifieurs (vs indépendants)
USE_PSEUDO   = True     # pseudo-étiquetage semi-supervisé
USE_SMOTE    = True     # rééquilibrage SMOTE par label

print("Imports OK")
print(f"Config : SAMPLE_SIZE={SAMPLE_SIZE}  FAST={FAST}  DROP_LOCATION={DROP_LOCATION}")
print(f"         USE_CHAIN={USE_CHAIN}  USE_PSEUDO={USE_PSEUDO}  USE_SMOTE={USE_SMOTE}")
print(f"         {len(PFAS_TARGET_COLS)} PFAS cibles")

Imports OK
Config : SAMPLE_SIZE=15000  FAST=True  DROP_LOCATION=True
         USE_CHAIN=True  USE_PSEUDO=True  USE_SMOTE=True
         27 PFAS cibles


## 1. Chargement des données

In [25]:
df = pd.read_parquet(PROCESSED_DIR / "CA-PFAS-ASGWS.parquet")
print(f"Dataset complet : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Sous-échantillon  : {df.shape[0]:,} lignes (exploration)")
df.head(3)

Dataset complet : 46,338 lignes × 201 colonnes
Sous-échantillon  : 15,000 lignes (exploration)


,gm_well_id,collection_date,ADONA_ngL,F53B_major_ngL,F53B_minor_ngL,FTS_4_2_ngL,FTS_6_2_ngL,FTS_8_2_ngL,HFPO_DA_ngL,NEtFOSAA_ngL,...,runoff_mm,soil_moi_0_10_kg_m2,soil_moi_10_40_kg_m2,soil_moi_40_100_kg_m2,soil_moi_100_200_kg_m2,root_zone_moist_kg_m2,temp_c,snowpack_mm,gldas_dist_km,soil_moisture_total_mm
0,CA4310012_034_034,2019-08-14,1.0,1.0,1.0,NaN,NaN,NaN,2.5,1.0,...,4.032258e-08,26.496033,83.483879,169.581467,321.262787,70.115906,21.815240,0.0,6.036,600.824165
1,CA1910152_009_009,2022-03-23,0.9,0.9,0.9,NaN,NaN,NaN,0.9,0.9,...,2.155645e-04,23.791937,71.348015,141.724136,240.720016,23.791937,17.314142,0.0,9.554,477.584105
2,CA1900046_006_006,2023-03-13,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,...,3.215951e-02,28.872391,87.252815,172.512527,268.661407,192.132141,9.892999,0.0,11.086,557.299141


## 2. Construction de la matrice de labels (seuils EPA 2024 par PFAS)

Pour chaque PFAS *k* et chaque échantillon *i* :

$$Y_{i,k} = \begin{cases}
1 & \text{si la concentration} > \text{seuil}_k \quad(\text{MCL EPA 2024 ou 2 ng/L})\\
0 & \text{si mesurée et} \le \text{seuil}_k\\
\text{NaN} & \text{si le PFAS } k \text{ n'a pas été mesuré dans ce programme}
\end{cases}$$

Les NaN sont la clé du caractère **semi-supervisé** : un même puits peut être étiqueté
pour PFOS mais non étiqueté pour PFPeS, selon le panel analytique du programme.

In [26]:
# Seuils appliqués
seuils = pd.DataFrame([
    {"PFAS": _label_name(c),
     "seuil_ngL": DETECTION_THRESHOLDS.get(c, DEFAULT_THRESHOLD),
     "source": "EPA 2024 NPDWR" if DETECTION_THRESHOLDS.get(c, DEFAULT_THRESHOLD) != DEFAULT_THRESHOLD else "2 ng/L (MDL)"}
    for c in PFAS_TARGET_COLS
])
print("Seuils réglementaires par composé :")
display(seuils[seuils.source == "EPA 2024 NPDWR"])

# Construction de Y
Y = build_label_matrix(df)
print(f"\nMatrice de labels Y : {Y.shape[0]:,} × {Y.shape[1]}")

Seuils réglementaires par composé :


,PFAS,seuil_ngL,source
0,PFOS,4.0,EPA 2024 NPDWR
3,PFOA,4.0,EPA 2024 NPDWR
9,PFHxS,10.0,EPA 2024 NPDWR
23,PFNA,10.0,EPA 2024 NPDWR



Matrice de labels Y : 15,000 × 27


In [27]:
# Tableau récapitulatif : prévalence + couverture (étiquetés vs non étiquetés)
recap = []
for c in PFAS_TARGET_COLS:
    n_lbl = int(Y[c].notna().sum())
    n_pos = int(Y[c].sum())
    recap.append({
        "PFAS":        _label_name(c),
        "seuil_ngL":   DETECTION_THRESHOLDS.get(c, DEFAULT_THRESHOLD),
        "étiquetés":   n_lbl,
        "non_étiq":    len(Y) - n_lbl,
        "positifs":    n_pos,
        "prévalence%": round(100 * Y[c].mean(), 1),
    })
recap_df = pd.DataFrame(recap)
recap_df

,PFAS,seuil_ngL,étiquetés,non_étiq,positifs,prévalence%
0,PFOS,4.0,14974,26,6040,40.3
1,PFBS,2.0,14257,743,5880,41.2
2,PFHxA,2.0,14353,647,5754,40.1
3,PFOA,4.0,14970,30,5281,35.3
4,FTS_6_2,2.0,8329,6671,4153,49.9
5,PFHpA,2.0,14582,418,4138,28.4
6,PFBA,2.0,8309,6691,4060,48.9
7,FTS_8_2,2.0,8329,6671,3619,43.5
8,PFPeA,2.0,8275,6725,3501,42.3
9,PFHxS,10.0,14266,734,2234,15.7


In [28]:
# Visualisation : prévalence par PFAS, colorée selon la source du seuil
fig, ax = plt.subplots(figsize=(10, 8))
order = recap_df.sort_values("prévalence%", ascending=True)
colors = ["#c0392b" if DETECTION_THRESHOLDS.get(p+"_ngL", DEFAULT_THRESHOLD) != DEFAULT_THRESHOLD
          else "#2980b9" for p in order["PFAS"]]
ax.barh(order["PFAS"], order["prévalence%"], color=colors)
ax.set_xlabel("Prévalence (% des échantillons dépassant le seuil)")
ax.set_title("Prévalence des dépassements par PFAS\n(rouge = MCL EPA 2024, bleu = 2 ng/L)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Structure semi-supervisée — couverture des mesures

Tous les programmes GAMA ne mesurent pas le même panel. Le graphique ci-dessous
montre, pour chaque PFAS, la part d'échantillons **étiquetés** (mesurés) vs
**non étiquetés** (NaN, candidats au pseudo-étiquetage).

In [29]:
fig, ax = plt.subplots(figsize=(10, 8))
order = recap_df.sort_values("étiquetés", ascending=True)
ax.barh(order["PFAS"], order["étiquetés"], color="#27ae60", label="étiquetés (mesurés)")
ax.barh(order["PFAS"], order["non_étiq"], left=order["étiquetés"],
        color="#bdc3c7", label="non étiquetés (NaN → pseudo-label)")
ax.set_xlabel("Nombre d'échantillons")
ax.set_title("Couverture analytique : étiquetés vs candidats au pseudo-étiquetage")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

frac_unlabeled = recap_df["non_étiq"].sum() / (recap_df["étiquetés"].sum() + recap_df["non_étiq"].sum())
print(f"Fraction globale de cellules non étiquetées (NaN) : {100*frac_unlabeled:.1f}%")
print("→ C'est ce signal que la semi-supervision exploite via le pseudo-étiquetage.")

Fraction globale de cellules non étiquetées (NaN) : 28.9%
→ C'est ce signal que la semi-supervision exploite via le pseudo-étiquetage.


## 4. Ingénierie des features et retrait de la localisation

`build_feature_matrix(df, drop_location=True)` :
- ajoute les features temporelles (year, month, season) ;
- exclut les colonnes de **fuite** (concentrations `_ngL`, labels) ;
- avec `drop_location=True` (protocole Dong), retire les **identifiants géographiques
  purs** mais **conserve la proximité aux sources**.

In [30]:
X, num_cols, cat_cols = build_feature_matrix(df, drop_location=DROP_LOCATION)
feature_names = num_cols + cat_cols
print(f"Matrice X : {X.shape[0]:,} × {X.shape[1]} features "
      f"({len(num_cols)} numériques + {len(cat_cols)} catégorielles)\n")

# Localisation retirée
loc_removed = sorted(LOCATION_FEATURES & set(df.columns)) if DROP_LOCATION else []
print("Localisation RETIRÉE (protocole Dong et al.) :")
for c in loc_removed:
    print(f"   ✗ {c}")

# Proximité conservée
print("\nProximité aux sources CONSERVÉE (mécanistique) :")
for c in feature_names:
    if "geotracker" in c.lower():
        print(f"   ✓ {c}")

Matrice X : 15,000 × 97 features (93 numériques + 4 catégorielles)

Localisation RETIRÉE (protocole Dong et al.) :
   ✗ county
   ✗ dwr_basin
   ✗ dwr_region
   ✗ latitude
   ✗ longitude
   ✗ regional_board
   ✗ sgma_basin_name
   ✗ sgma_region_office
   ✗ sgma_subbasin_name

Proximité aux sources CONSERVÉE (mécanistique) :
   ✓ dist_geotracker_km
   ✓ n_geotracker_within_1km
   ✓ n_geotracker_within_3km
   ✓ n_geotracker_within_10km
   ✓ n_geotracker_within_50km
   ✓ nearest_geotracker_type


In [31]:
# Répartition par groupe thématique
groups = {
    "GEOTRACKER (sources)": [c for c in num_cols if "geotracker" in c],
    "GLDAS (hydro-climat)": [c for c in num_cols if any(x in c for x in
        ["rainfall","et_mm","runoff","soil_moi","root_zone","temp_c","snowpack","soil_moisture_total"])],
    "SSURGO (sol)":         [c for c in num_cols if c.startswith("soil_")],
    "AQS (air)":            [c for c in num_cols if c.startswith("aqs_")],
    "CO_CONTAM (polluants)":[c for c in num_cols if c.startswith("cocontam_")],
    "TEMPOREL":             [c for c in num_cols if c in ["year","month","season"]],
    "CATÉGORIEL":           cat_cols,
}
print("Groupes de features retenues :")
for g, cols in groups.items():
    print(f"  {g:<24} {len(cols):3d}")

Groupes de features retenues :
  GEOTRACKER (sources)       5
  GLDAS (hydro-climat)      11
  SSURGO (sol)              25
  AQS (air)                  8
  CO_CONTAM (polluants)     44
  TEMPOREL                   3
  CATÉGORIEL                 4


## 5. Prétraitement et split train/test

Split **stratifié sur PFOS** (le label pivot le plus fréquent) pour préserver
la distribution. Imputation médiane (numériques) + encodage ordinal (catégorielles).

In [32]:
pivot_y = Y["PFOS_ngL"].fillna(-1).values
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, stratify=pivot_y, random_state=RANDOM_STATE,
)
Y_train = Y_train.reset_index(drop=True)
Y_test  = Y_test.reset_index(drop=True)
print(f"Train : {len(Y_train):,}   |   Test : {len(Y_test):,}")

preprocessor = build_preprocessor(num_cols, cat_cols)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
print(f"X_train_proc : {X_train_proc.shape}")

Train : 12,000   |   Test : 3,000
X_train_proc : (12000, 97)


## 6. Pourquoi une *chaîne* de classifieurs ? — co-occurrence des PFAS

Les PFAS partagent des sources communes (sites industriels, bases militaires,
mousses anti-incendie). Ils **co-occurrent** donc fortement. Une chaîne de
classifieurs exploite cette structure : le modèle du PFAS *k* reçoit en entrée
les probabilités prédites pour les PFAS *1…k−1*.

In [33]:
# Corrélation de Pearson entre labels sur les échantillons à panel complet
full_mask = ~np.any(np.isnan(Y[PFAS_TARGET_COLS].values), axis=1)
Y_full = Y[PFAS_TARGET_COLS][full_mask].astype(float)
Y_full.columns = [_label_name(c) for c in PFAS_TARGET_COLS]
corr = Y_full.corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": 0.6, "label": "Corrélation de Pearson"},
            xticklabels=True, yticklabels=True, ax=ax)
ax.set_title(f"Co-occurrence des PFAS détectés (panel complet, n={full_mask.sum():,})")
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7, rotation=0)
plt.tight_layout()
plt.show()

# Paires les plus corrélées
pairs = (corr.where(~np.eye(len(corr), dtype=bool)).stack()
         .sort_values(ascending=False))
print("Paires de PFAS les plus co-occurrentes :")
seen = set()
for (a, b), v in pairs.items():
    key = frozenset((a, b))
    if key not in seen:
        seen.add(key)
        print(f"  {a:<10} ↔ {b:<10}  r = {v:.3f}")
    if len(seen) >= 8:
        break

Paires de PFAS les plus co-occurrentes :
  F53B_major ↔ F53B_minor  r = 1.000
  NEtFOSAA   ↔ NMeFOSAA    r = 0.994
  F53B_major ↔ ADONA       r = 0.993
  F53B_minor ↔ ADONA       r = 0.993
  PFTrDA     ↔ PFDS        r = 0.982
  PFDoDA     ↔ PFTrDA      r = 0.973
  PFDS       ↔ PFDoDA      r = 0.954
  PFTeDA     ↔ PFTrDA      r = 0.931


## 7. Entraînement de la chaîne semi-supervisée

`fit_chain(...)` enchaîne, pour chaque PFAS (par prévalence décroissante) :

1. **Modèle XGBoost initial** sur les échantillons étiquetés (+ SMOTE) ;
2. **Pseudo-étiquetage** des échantillons non mesurés dont la confiance dépasse
   les seuils (`P ≥ 0.95` → 1, `P ≤ 0.05` → 0) ;
3. **Réentraînement** sur données étiquetées + pseudo-étiquetées ;
4. les probabilités prédites alimentent les features du PFAS suivant (**chaîne**).

> ⏱️ Cellule la plus lourde. Avec `FAST=True` et `SAMPLE_SIZE=15000`, comptez ~3–6 min.

In [34]:
%%time
models, train_proba = fit_chain(
    X_train_proc, Y_train,
    use_pseudo=USE_PSEUDO,
    use_chain=USE_CHAIN,
    use_smote=USE_SMOTE,
    fast=FAST,
)
print(f"\n{len([m for m, _ in models if m is not None])} modèles entraînés.")


27 modèles entraînés.
CPU times: user 2min 13s, sys: 1.56 s, total: 2min 14s
Wall time: 1min 9s


In [35]:
# Prédiction sur le test set
proba_test = predict_chain(models, X_test_proc, use_chain=USE_CHAIN)
metrics = evaluate_multilabel(Y_test, proba_test)
print("Évaluation terminée.")

Évaluation terminée.


## 8. Résultats par label (AUROC, F1, prévalence)

In [36]:
per_label = pd.DataFrame(metrics["per_label"]).sort_values("roc_auc", ascending=False)
per_label_disp = per_label.copy()
per_label_disp["prevalence"] = (100 * per_label_disp["prevalence"]).round(1)
per_label_disp.columns = ["PFAS", "n_étiq", "prév%", "AUROC", "AvgPrec", "F1"]
per_label_disp.reset_index(drop=True)

,PFAS,n_étiq,prév%,AUROC,AvgPrec,F1
0,FTS_4_2,1627,14.3,0.9843,0.9198,0.8255
1,ADONA,2621,4.6,0.9809,0.8033,0.7063
2,NMeFOSAA,1757,16.1,0.9783,0.9102,0.8381
3,NEtFOSAA,1763,16.7,0.9763,0.9060,0.8291
4,PFNA,2914,3.8,0.9733,0.6196,0.5980
5,PFHpS,1618,9.2,0.9603,0.7090,0.6854
6,FTS_8_2,1629,42.4,0.9600,0.9423,0.8723
7,FTS_6_2,1629,48.6,0.9581,0.9561,0.8919
8,F53B_major,2615,3.5,0.9553,0.6824,0.6168
9,PFPeS,1635,17.3,0.9492,0.8218,0.7346


In [37]:
# Barplot AUROC par label
pl = per_label.dropna(subset=["roc_auc"]).sort_values("roc_auc")
fig, ax = plt.subplots(figsize=(9, 9))
cmap = plt.cm.RdYlGn((pl["roc_auc"].values - 0.5) / 0.5)
ax.barh(pl["label"], pl["roc_auc"], color=cmap)
ax.axvline(0.5,  color="gray", ls="--", lw=1, label="aléatoire")
ax.axvline(0.75, color="#34495e", ls=":", lw=1, label="0.75")
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("AUROC (test set)")
ax.set_title("Performance par PFAS — chaîne semi-supervisée")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUROC : min={pl.roc_auc.min():.3f}  médiane={pl.roc_auc.median():.3f}  max={pl.roc_auc.max():.3f}")

AUROC : min=0.780  médiane=0.942  max=0.984


## 9. Métriques globales multilabel

- **Hamming loss** : fraction de labels mal prédits (↓ meilleur)
- **Exact Match Ratio** : part d'échantillons dont *tous* les labels sont corrects (↑)
- **Macro-AUROC** : AUROC moyen sur les labels (chaque PFAS pèse autant)
- **Micro-F1** : F1 agrégé sur toutes les paires (échantillon, label)

In [38]:
g = metrics["global"]
print("══════════════════════════════════════════════")
print("  MÉTRIQUES GLOBALES — Test set")
print("══════════════════════════════════════════════")
print(f"  Hamming loss      : {g.get('hamming_loss')}")
print(f"  Exact Match Ratio : {g.get('exact_match_ratio')}")
print(f"  Macro-AUROC       : {g.get('macro_roc_auc')}")
print(f"  Micro-F1          : {g.get('micro_f1')}")
print(f"  (panel complet n  : {g.get('n_full_panel'):,})")
print("══════════════════════════════════════════════")

══════════════════════════════════════════════
  MÉTRIQUES GLOBALES — Test set
══════════════════════════════════════════════
  Hamming loss      : 0.2454
  Exact Match Ratio : 0.1194
  Macro-AUROC       : 0.9343
  Micro-F1          : 0.7887
  (panel complet n  : 67)
══════════════════════════════════════════════


## 10. Validation croisée 5-fold

Validation croisée stratifiée (sur PFOS) pour estimer la stabilité des métriques
globales. ⏱️ ~5× le temps d'un entraînement.

In [39]:
%%time
fold_results = cross_validate_chain(
    X_train_proc, Y_train,
    use_pseudo=USE_PSEUDO, use_chain=USE_CHAIN, use_smote=USE_SMOTE, fast=FAST,
)
cv_df = pd.DataFrame(fold_results)
cv_df

CPU times: user 9min 47s, sys: 4.73 s, total: 9min 51s
Wall time: 5min 1s


,fold,hamming_loss,emr,macro_auc,micro_f1
0,1,0.2963,0.0984,0.9246,0.7705
1,2,0.2826,0.1096,0.9161,0.7722
2,3,0.2079,0.1591,0.9222,0.7805
3,4,0.2178,0.1525,0.9175,0.7732
4,5,0.2454,0.1500,0.9231,0.7772


In [40]:
# Synthèse CV + graphique
metrics_cv = {"hamming_loss": "Hamming loss", "emr": "Exact Match",
              "macro_auc": "Macro-AUROC", "micro_f1": "Micro-F1"}
colors = {"hamming_loss": "#e74c3c", "emr": "#2ecc71",
          "macro_auc": "#3498db", "micro_f1": "#f39c12"}
fig, ax = plt.subplots(figsize=(8, 4))
for k, lbl in metrics_cv.items():
    vals = cv_df[k]
    ax.plot(cv_df["fold"], vals, "o-", color=colors[k],
            label=f"{lbl} ({vals.mean():.3f}±{vals.std():.3f})")
    ax.axhline(vals.mean(), ls="--", color=colors[k], alpha=0.4, lw=1)
ax.set_xticks(cv_df["fold"])
ax.set_xlabel("Fold")
ax.set_title(f"Validation croisée {CV_FOLDS}-fold")
ax.legend(fontsize=9, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Étude d'ablation — justifier les choix méthodologiques

On quantifie l'apport de chaque ingrédient du protocole sur un sous-échantillon
rapide (modèles XGBoost indépendants par label pour aller vite). Deux questions :

1. **Retrait de la localisation** : combien perd-on en AUROC ? (et le modèle
   s'appuyait-il dessus ?)
2. **Chaîne vs indépendants** : la chaîne aide-t-elle ?

In [41]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

ABLATION_LABELS = ["PFOS_ngL", "PFOA_ngL", "PFHxS_ngL", "PFBS_ngL", "PFHxA_ngL", "PFNA_ngL"]
FAST_XGB = dict(n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, eval_metric="auc", n_jobs=4,
                random_state=RANDOM_STATE, verbosity=0)

def quick_auc(drop_loc):
    Xa, ncol, ccol = build_feature_matrix(df, drop_location=drop_loc)
    pre = build_preprocessor(ncol, ccol)
    Xp = pre.fit_transform(Xa)
    feats = ncol + ccol
    aucs, imp_loc = {}, {}
    for col in ABLATION_LABELS:
        yv = Y[col].values
        mask = ~np.isnan(yv)
        yv = yv[mask].astype(int)
        Xv = Xp[mask]
        Xtr, Xte, ytr, yte = train_test_split(Xv, yv, test_size=0.2,
                                              stratify=yv, random_state=RANDOM_STATE)
        clf = XGBClassifier(**FAST_XGB,
                            scale_pos_weight=(yte==0).sum()/max((yte==1).sum(),1))
        clf.fit(Xtr, ytr)
        aucs[_label_name(col)] = roc_auc_score(yte, clf.predict_proba(Xte)[:,1])
        if not drop_loc:
            imp = pd.Series(clf.feature_importances_, index=feats)
            imp_loc[col] = imp
    return aucs, imp_loc

auc_with, imp_with = quick_auc(drop_loc=False)
auc_without, _     = quick_auc(drop_loc=True)

abl = pd.DataFrame({"AVEC localisation": auc_with, "SANS localisation": auc_without})
abl["Δ AUROC"] = abl["SANS localisation"] - abl["AVEC localisation"]
abl.loc["MOYENNE"] = abl.mean()
abl.round(4)

,AVEC localisation,SANS localisation,Δ AUROC
PFOS,0.9473,0.9414,-0.0059
PFOA,0.9372,0.9327,-0.0045
PFHxS,0.9620,0.9572,-0.0048
PFBS,0.9239,0.9201,-0.0038
PFHxA,0.9131,0.9081,-0.0049
PFNA,0.9746,0.9773,0.0027
MOYENNE,0.9430,0.9395,-0.0035


In [42]:
# Où se classe la localisation quand on la garde ? (sur PFOS)
loc_feats = [c for c in LOCATION_FEATURES if c in imp_with["PFOS_ngL"].index]
imp_pfos = imp_with["PFOS_ngL"].sort_values(ascending=False).reset_index()
imp_pfos.columns = ["feature", "importance"]
imp_pfos["rang"] = imp_pfos.index + 1

print("Rang des features de localisation quand INCLUSES (PFOS) :")
for f in sorted(loc_feats):
    row = imp_pfos[imp_pfos.feature == f]
    if len(row):
        r = row.iloc[0]
        print(f"   {f:<22} rang #{int(r['rang']):<3} importance={r['importance']:.4f}")
tot = imp_with["PFOS_ngL"][loc_feats].sum()
print(f"\nImportance cumulée de la localisation : {tot:.1%} du total (PFOS)")
print("\nTop 5 features (avec localisation) :")
for _, r in imp_pfos.head(5).iterrows():
    flag = " ← LOCALISATION" if r["feature"] in loc_feats else ""
    print(f"   #{int(r['rang'])} {r['feature']:<26} {r['importance']:.4f}{flag}")

Rang des features de localisation quand INCLUSES (PFOS) :
   county                 rang #5   importance=0.0220
   dwr_region             rang #68  importance=0.0070
   latitude               rang #8   importance=0.0181
   longitude              rang #14  importance=0.0144
   regional_board         rang #1   importance=0.0583
   sgma_region_office     rang #10  importance=0.0161

Importance cumulée de la localisation : 13.6% du total (PFOS)

Top 5 features (avec localisation) :
   #1 regional_board             0.0583 ← LOCALISATION
   #2 n_geotracker_within_50km   0.0489
   #3 cocontam_dca12             0.0341
   #4 gm_dataset_name            0.0251
   #5 county                     0.0220 ← LOCALISATION


**Lecture.** Si le Δ AUROC est faible (~−0.5 pt) **alors que** la localisation
domine l'importance quand on la garde, c'est la preuve que le modèle *mémorisait
l'espace* sans coût prédictif réel — on a donc raison de la retirer pour
généraliser aux zones non surveillées (cf. Dong et al.).

## 12. Comparaison des 3 approches de division en classes

Dong et al. répartissent les composés en **4 classes AVANT l'entraînement**. Point
crucial souvent mal compris : ces classes ne sont **pas des familles chimiques**
(PFCA/PFSA/FTS…) mais des **paliers de disponibilité des données** (nombre
d'observations). Nos 27 PFAS tombent naturellement en 4 paliers de couverture
quasi identiques à ceux du papier.

On compare ici **3 stratégies** sur le *même* split train/test :

| Approche | Principe |
|---|---|
| **`global`** | Chaîne unique ordonnée par prévalence, pseudo-étiquetage uniforme (déjà entraînée au §7) |
| **`nested`** | **1 chaîne** ordonnée par palier + pseudo-étiquetage **calibré par palier** |
| **`class`** | **4 chaînes indépendantes**, une par palier — **réplication littérale Dong et al.** |

**Clé méthodologique** : la couverture est *parfaitement emboîtée* (quand un PFAS
rare est mesuré, PFOS l'est dans ~100 % des cas). Cela autorise l'approche `nested`
à conserver les corrélations **inter-paliers** (ex. PFOS↔PFDS), que les 4 chaînes
isolées de `class` perdent.

In [43]:
# Paliers de couverture + politique de semi-supervision par palier
print("PALIERS DE COUVERTURE (≈ 'Classes' de Dong et al. — par disponibilité, pas chimie)")
print("="*78)
for tier, cols in COVERAGE_TIERS.items():
    pol = TIER_PSEUDO_POLICY[tier]
    noms = ", ".join(_label_name(c) for c in cols[:7]) + (" …" if len(cols) > 7 else "")
    print(f"  Palier {tier} ({len(cols):2d} PFAS) | pseudo={str(pol['use_pseudo']):5} "
          f"conf≥{pol['conf_high']} | {noms}")
print()
print("Justification du calibrage :")
print("  Palier 0/1 : ~40-46k étiquetés, peu de non-étiquetés → pseudo inutile (off)")
print("  Palier 2   : ~26k étiq. / ~20k non-étiq. → sweet-spot Dong (+2.9%) → on")
print("  Palier 3   : ~6-20k étiq. / bcp de non-étiq. → on mais CONSERVATEUR (P≥0.99)")

PALIERS DE COUVERTURE (≈ 'Classes' de Dong et al. — par disponibilité, pas chimie)
  Palier 0 (10 PFAS) | pseudo=False conf≥0.95 | PFOS, PFOA, PFNA, PFHpA, PFUnDA, PFDA, PFHxA …
  Palier 1 ( 3 PFAS) | pseudo=False conf≥0.95 | ADONA, F53B_major, F53B_minor
  Palier 2 (11 PFAS) | pseudo=True  conf≥0.95 | PFTeDA, PFTrDA, NEtFOSAA, NMeFOSAA, PFPeS, FTS_8_2, FTS_6_2 …
  Palier 3 ( 3 PFAS) | pseudo=True  conf≥0.99 | NFDHA, PFOSAm, PFDS

Justification du calibrage :
  Palier 0/1 : ~40-46k étiquetés, peu de non-étiquetés → pseudo inutile (off)
  Palier 2   : ~26k étiq. / ~20k non-étiq. → sweet-spot Dong (+2.9%) → on
  Palier 3   : ~6-20k étiq. / bcp de non-étiq. → on mais CONSERVATEUR (P≥0.99)


### 12.1 Entraînement des approches `nested` et `class`

L'approche `global` a déjà été entraînée au §7 (variable `models`). On entraîne
ici les deux autres sur le **même** `X_train_proc` / `Y_train`.

> ⏱️ Avec `FAST=True` et `SAMPLE_SIZE=15000`, comptez ~6-10 min (2 × ~27 modèles).

In [44]:
%%time
# global : réutilise la chaîne déjà entraînée au §7
proba_global = predict_chain(models, X_test_proc, use_chain=USE_CHAIN)

# nested : 1 chaîne emboîtée ordonnée par palier + semi-sup calibrée
print("── NESTED : chaîne emboîtée par palier ──")
models_nested, _ = fit_nested_chain(
    X_train_proc, Y_train, use_chain=USE_CHAIN, use_smote=USE_SMOTE, fast=FAST,
)
proba_nested = predict_chain(models_nested, X_test_proc, use_chain=USE_CHAIN)

# class : 4 chaînes indépendantes (réplication Dong et al.)
print("── CLASS : 4 chaînes indépendantes ──")
class_models = fit_class_chains(
    X_train_proc, Y_train, use_chain=USE_CHAIN, use_smote=USE_SMOTE, fast=FAST,
)
proba_class = predict_class_chains(class_models, X_test_proc, use_chain=USE_CHAIN)

print("\nApproches entraînées : global, nested, class")

── NESTED : chaîne emboîtée par palier ──
── CLASS : 4 chaînes indépendantes ──

Approches entraînées : global, nested, class
CPU times: user 3min 11s, sys: 1.78 s, total: 3min 13s
Wall time: 1min 37s


### 12.2 Métriques globales comparées

In [45]:
proba_by = {"global": proba_global, "nested": proba_nested, "class": proba_class}

rows = []
for name, proba in proba_by.items():
    g = evaluate_multilabel(Y_test, proba)["global"]
    rows.append({
        "approche":    name,
        "macro_AUC":   g.get("macro_roc_auc"),
        "micro_F1":    g.get("micro_f1"),
        "Hamming":     g.get("hamming_loss"),
        "ExactMatch":  g.get("exact_match_ratio"),
    })
comp_df = pd.DataFrame(rows).set_index("approche")
best = comp_df["macro_AUC"].idxmax()
print(f"Meilleure macro-AUROC : {best}  ({comp_df.loc[best, 'macro_AUC']:.4f})")
comp_df.style.highlight_max(subset=["macro_AUC", "micro_F1", "ExactMatch"], color="#c8e6c9")\
             .highlight_min(subset=["Hamming"], color="#c8e6c9")

Meilleure macro-AUROC : class  (0.9424)


,macro_AUC,micro_F1,Hamming,ExactMatch
approche,,,,
global,0.934300,0.788700,0.245400,0.119400
nested,0.930100,0.783300,0.249300,0.104500
class,0.942400,0.785300,0.239900,0.089600


### 12.3 Macro-AUROC **par palier** — le point décisif

C'est ici que les approches divergent : sur les **PFAS rares** (palier 3),
isoler la chaîne (`class`) ou calibrer la semi-supervision (`nested`) peut éviter
que les probabilités amont peu fiables n'injectent du bruit.

In [46]:
tier_rows = []
for name, proba in proba_by.items():
    for tier, tm in evaluate_by_tier(Y_test, proba).items():
        tier_rows.append({"approche": name, "palier": tier,
                          "macro_AUC": tm.get("macro_roc_auc")})
tier_df = (pd.DataFrame(tier_rows)
           .pivot(index="palier", columns="approche", values="macro_AUC"))
tier_df = tier_df[["global", "nested", "class"]]
display(tier_df.style.highlight_max(axis=1, color="#c8e6c9").format("{:.4f}"))

approche,global,nested,class
palier,,,
0,0.9354,0.9378,0.9378
1,0.9610,0.9588,0.9781
2,0.9511,0.9395,0.9517
3,0.8418,0.8410,0.8875


In [47]:
# Graphique comparatif (global + par palier)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
palette = {"global": "#7f8c8d", "nested": "#2980b9", "class": "#e67e22"}

# (1) métriques globales
metric_cols = ["macro_AUC", "micro_F1", "ExactMatch"]
x = np.arange(len(metric_cols)); w = 0.25
for i, name in enumerate(comp_df.index):
    ax1.bar(x + i*w, comp_df.loc[name, metric_cols].values, w,
            label=name, color=palette[name])
ax1.set_xticks(x + w); ax1.set_xticklabels(metric_cols)
ax1.set_ylim(0, 1); ax1.set_title("Métriques globales (test set)")
ax1.legend(); ax1.grid(axis="y", alpha=0.3)

# (2) macro-AUROC par palier
x2 = np.arange(len(tier_df.index))
for i, name in enumerate(tier_df.columns):
    ax2.bar(x2 + i*w, tier_df[name].values, w, label=name, color=palette[name])
ax2.set_xticks(x2 + w); ax2.set_xticklabels([f"Palier {t}" for t in tier_df.index])
ax2.set_ylim(0.5, 1.0); ax2.set_title("Macro-AUROC par palier de couverture")
ax2.legend(); ax2.grid(axis="y", alpha=0.3)

fig.suptitle("Comparaison des 3 approches de division en classes", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ml_multilabel_approaches_nb.png", dpi=150, bbox_inches="tight")
plt.show()

**Lecture pour le mémoire.**

- Si `class` (4 chaînes isolées) domine surtout sur le **palier 3** (PFAS rares),
  c'est l'argument exact de Dong et al. : isoler les composés peu mesurés évite
  qu'une longue chaîne amont peu fiable ne propage du bruit.
- Si `nested` rattrape ou dépasse `class`, c'est que l'**emboîtement** de la
  couverture (rare mesuré ⇒ fréquent mesuré) permet de garder les corrélations
  inter-paliers **sans** le coût du bruit — une amélioration du protocole adaptée
  à nos données.
- L'écart sur paliers 0/1 (PFAS fréquents) est généralement faible : abondance de
  données → le choix d'architecture compte peu.

⚠️ Sur sous-échantillon (`SAMPLE_SIZE=15000`) les écarts sont bruités ; relancer
avec `SAMPLE_SIZE=None, FAST=False` pour les chiffres définitifs. Équivalent en
ligne de commande : `python -m src.ml_multilabel --approach compare`.

## 13. Importance des features — SHAP (optionnel, lent)

Mettre `RUN_SHAP = True` pour calculer les valeurs SHAP des labels les plus
fréquents et identifier les drivers environnementaux.

In [48]:
RUN_SHAP = False   # ← passer à True pour activer (~5-10 min)

if RUN_SHAP:
    import shap
    top_labels = PFAS_TARGET_COLS[:3]
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(len(X_test_proc), min(800, len(X_test_proc)), replace=False)
    for k, (model, col) in enumerate(models):
        if col not in top_labels or model is None:
            continue
        Xs = X_test_proc[idx]
        sv = shap.TreeExplainer(model).shap_values(Xs)
        sv = sv[1] if isinstance(sv, list) else sv
        nf = min(len(feature_names), sv.shape[1])
        shap.summary_plot(sv[:, :nf], Xs[:, :nf],
                          feature_names=feature_names[:nf],
                          max_display=15, show=False)
        plt.title(f"SHAP — {_label_name(col)}")
        plt.tight_layout(); plt.show()
else:
    print("SHAP désactivé (RUN_SHAP=False).")

SHAP désactivé (RUN_SHAP=False).


## 14. Sauvegarde des artefacts

In [49]:
# Modèle
artifact = {
    "models": models, "preprocessor": preprocessor,
    "feature_names": feature_names, "num_cols": num_cols, "cat_cols": cat_cols,
    "pfas_targets": PFAS_TARGET_COLS, "detection_thresholds": DETECTION_THRESHOLDS,
    "use_chain": USE_CHAIN, "use_pseudo": USE_PSEUDO, "drop_location": DROP_LOCATION,
}
with open(MODELS_DIR / "chain_multilabel_nb.pkl", "wb") as f:
    pickle.dump(artifact, f)

# Métriques
per_label.to_csv(PROCESSED_DIR / "ml_multilabel_per_label_nb.csv", index=False)
cv_df.to_csv(PROCESSED_DIR / "ml_multilabel_cv_nb.csv", index=False)
results = {
    "protocol": "Dong et al. 2024 — Chain + XGBoost + SMOTE + Semi-supervised",
    "n_pfas_targets": len(PFAS_TARGET_COLS),
    "sample_size": int(len(df)), "drop_location": DROP_LOCATION,
    "use_chain": USE_CHAIN, "use_pseudo": USE_PSEUDO,
    "test": metrics["global"],
    "cv_mean": {k: round(float(cv_df[k].mean()), 4)
                for k in ["hamming_loss", "emr", "macro_auc", "micro_f1"]},
}
with open(PROCESSED_DIR / "ml_multilabel_results_nb.json", "w") as f:
    json.dump(results, f, indent=2)
print("Artefacts sauvegardés :")
print("  models/chain_multilabel_nb.pkl")
print("  data/processed/ml_multilabel_per_label_nb.csv")
print("  data/processed/ml_multilabel_results_nb.json")

Artefacts sauvegardés :
  models/chain_multilabel_nb.pkl
  data/processed/ml_multilabel_per_label_nb.csv
  data/processed/ml_multilabel_results_nb.json


## 15. Synthèse finale

In [50]:
g = metrics["global"]
print("╔══════════════════════════════════════════════════════════╗")
print("║   MULTILABEL PFAS — Protocole Dong et al. (2024)          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Échantillons        : {len(df):>8,}                          ║")
print(f"║  PFAS cibles         : {len(PFAS_TARGET_COLS):>8}                          ║")
print(f"║  Features            : {len(feature_names):>8}  (localisation retirée)  ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Macro-AUROC (test)  : {g.get('macro_roc_auc'):>8}                          ║")
print(f"║  Micro-F1   (test)   : {g.get('micro_f1'):>8}                          ║")
print(f"║  Hamming loss (test) : {g.get('hamming_loss'):>8}                          ║")
print(f"║  Exact Match (test)  : {g.get('exact_match_ratio'):>8}                          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  CV macro-AUROC      : {cv_df['macro_auc'].mean():.3f} ± {cv_df['macro_auc'].std():.3f}                  ║")
print("╚══════════════════════════════════════════════════════════╝")
print()
print("Ingrédients du protocole :")
print(f"  • Seuils EPA 2024 NPDWR par PFAS (PFOA/PFOS=4, PFHxS/PFNA=10 ng/L)")
print(f"  • Chaîne de classifieurs    : {USE_CHAIN}")
print(f"  • Pseudo-étiquetage (semi-sup): {USE_PSEUDO}  (seuils {PSEUDO_LOW}/{PSEUDO_HIGH})")
print(f"  • SMOTE par label           : {USE_SMOTE}")
print(f"  • Retrait localisation      : {DROP_LOCATION}")

╔══════════════════════════════════════════════════════════╗
║   MULTILABEL PFAS — Protocole Dong et al. (2024)          ║
╠══════════════════════════════════════════════════════════╣
║  Échantillons        :   15,000                          ║
║  PFAS cibles         :       27                          ║
║  Features            :       97  (localisation retirée)  ║
╠══════════════════════════════════════════════════════════╣
║  Macro-AUROC (test)  :   0.9343                          ║
║  Micro-F1   (test)   :   0.7887                          ║
║  Hamming loss (test) :   0.2454                          ║
║  Exact Match (test)  :   0.1194                          ║
╠══════════════════════════════════════════════════════════╣
║  CV macro-AUROC      : 0.921 ± 0.004                  ║
╚══════════════════════════════════════════════════════════╝

Ingrédients du protocole :
  • Seuils EPA 2024 NPDWR par PFAS (PFOA/PFOS=4, PFHxS/PFNA=10 ng/L)
  • Chaîne de classifieurs    : True
  • Pseudo-éti